In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df=pd.read_csv('spam.csv',encoding='latin-1')

In [ ]:
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [ ]:
x=df['v2']
y=df['v1']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

In [ ]:
param_grid = {
    'model__alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
    'model__force_alpha': [True, False],
    'model__fit_prior': [True, False],
    'model__class_prior': [None]
}
print(param_grid)

{'model__alpha': [0.01, 0.1, 0.5, 1.0, 2.0], 'model__force_alpha': [True, False], 'model__fit_prior': [True, False], 'model__class_prior': [None]}


In [26]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('model', MultinomialNB())
])

param_grid = {
    'tfidf__max_features': [None, 1000, 5000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'model__alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
    'model__force_alpha': [True, False],
    'model__fit_prior': [True, False],
    'model__class_prior': [None]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, verbose=2, n_jobs=-1)


grid.fit(x_train, y_train)

Fitting 5 folds for each of 120 candidates, totalling 600 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('model', MultinomialNB())]),
             n_jobs=-1,
             param_grid={'model__alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
                         'model__class_prior': [None],
                         'model__fit_prior': [True, False],
                         'model__force_alpha': [True, False],
                         'tfidf__max_features': [None, 1000, 5000],
                         'tfidf__ngram_range': [(1, 1), (1, 2)]},
             verbose=2)

In [27]:
print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'model__alpha': 0.01, 'model__class_prior': None, 'model__fit_prior': True, 'model__force_alpha': True, 'tfidf__max_features': 5000, 'tfidf__ngram_range': (1, 2)}
Best Score: 0.9874346353419596


In [28]:
y_pred = grid.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Best Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Best Model Accuracy: 0.9785

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       965
        spam       0.97      0.87      0.92       150

    accuracy                           0.98      1115
   macro avg       0.97      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [29]:
text="Free Msg: Your USPS delivery [ID-99214] is on hold due to an incomplete address. Please update your details now to avoid return to sender: usps-tracking-update.com"
grid.predict([text])

array(['spam'], dtype='<U4')

In [31]:
import pickle
pickle.dump(grid,open("spam_detection.pkl","wb"))